# House Prices: Data Cleaning

Dataset: Kaggle "House Prices" (`train.csv`)

This notebook walks through cleaning the raw training data before we move on to encoding categorical variables in Step 2. Each section below has a short explanation followed by the code that performs it.

## Load the raw dataset

We start by loading `train.csv` and taking a quick look at its shape and data types.

In [11]:
import pandas as pd
pd.set_option('display.max_columns', 15)

df = pd.read_csv('train.csv')
print("Shape:", df.shape)
df.dtypes.value_counts()

Shape: (1460, 81)


object     43
int64      35
float64     3
Name: count, dtype: int64

## Inspect missing values

Before deciding how to handle missing data, we need to see which columns have `NaN`s and how many.

In [12]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1
dtype: int64

## Fill "feature does not exist" columns with `"None"`

According to the data dictionary (`data_description.txt`), for many columns a missing value doesn't mean the data is unknown, it means the house simply doesn't have that feature (e.g. no pool, no alley access, no fireplace, no garage, no basement, no masonry veneer).

For these columns, filling with the string `"None"` preserves that information instead of treating it as a data-quality problem.

In [13]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in none_cols:
    df[col] = df[col].fillna("None")

print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 349


## Impute genuinely missing values

The remaining missing values are real gaps in the data, not "feature absent" cases, so each needs a sensible imputation strategy:

- **`LotFrontage`** (259 missing): filled with the **median LotFrontage per Neighborhood**, since lot sizes vary systematically by area, a single global median would be less accurate.
- **`GarageYrBlt`** (81 missing): filled with `0`, since these rows have no garage (consistent with `GarageType = "None"` above).
- **`MasVnrArea`** (8 missing): filled with `0`, meaning no masonry veneer area.
- **`Electrical`** (1 missing): filled with the **mode** (most common value), since only a single row is affected.

In [14]:
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 0


## Verify cleaning and save the result

A final check confirms there are no missing values left. We then save the cleaned dataset as `train_cleaned.csv`, which we'll use as the input for the encoding steps.

In [16]:
assert df.isnull().sum().sum() == 0, "Missing values."

df.to_csv('train_cleaned.csv', index=False)
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,None,...,None,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,None,...,None,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,None,...,None,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,None,...,None,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,None,...,None,0,12,2008,WD,Normal,250000
